# Adaptive resolution — several refinement steps

Everything is defined at **corners**: the field is sampled at the corners of a regular grid, and a **cell** is the cube between 8 neighbouring corners.

1. Sample the field at the corners of a coarse grid.
2. Flag the cells whose 8 corners are not all on the same side of the surface.
3. Split each flagged cell into 8 children, evaluate the field at the children's corners, and keep only the children whose corners disagree. Repeat as many times as asked.
4. Split the remaining cells one last time (3 × 3 × 3 = 27 corners) and run marching cubes on each cell alone.
5. Sew the pieces together by merging the vertices that neighbouring cells computed twice.

Not handled yet:
- closing the lattice where it is cut by the box;
- a surface that only grazes a cell (all 8 corners on the same side) is dropped at that level, which can leave pinholes. Keeping the 26 neighbours of every kept cell fixes it, at 2 to 3 times the cost.

In [ ]:
import numpy as np
import trimesh
import plotly.graph_objects as go

import triply
from triply.utils import reload_all
reload_all()

## 1 — Parameters

In [ ]:
# ---- Box (mm) ----
box_size_x = 100
box_size_y = 100
box_size_z = 100

# ---- Coarse grid: number of cells along each axis ----
# The grid has one more corner than cells along each axis.
resolution_x = 15
resolution_y = 15
resolution_z = 15

# ---- Refinement ----
# How many times the cells containing the surface are halved.
# Final cell size = coarse cell size / 2**number_of_subdivisions.
number_of_subdivisions = 2
near_surface_factor = 0.05

# ---- Gyroid ----
period_x = 100      # size of one unit cell along x (mm)
period_y = 100
period_z = 100
thickness = 0.5     # band half-width, in field units (between 0 and about 1.43)

## 2 — Field on the coarse grid

In [ ]:
box_margin = 1e-12    # margin (mm) to the box faces: the gyroid is not computed in this margin

def distance_inside_box(x, y, z):
    # Distance (mm) to the nearest box face: positive inside the box, negative outside
    distance_x_face = np.minimum(x, box_size_x - x)
    distance_y_face = np.minimum(y, box_size_y - y)
    distance_z_face = np.minimum(z, box_size_z - z)
    print(distance_x_face.shape, distance_y_face.shape, distance_z_face.shape)
    distance = np.minimum(distance_x_face, distance_y_face)
    distance = np.minimum(distance, distance_z_face) - box_margin
    print(distance.shape)
    return distance


def compute_field(x, y, z, period_x, period_y, period_z, thickness):
    # Gyroid in band mode: positive (solid) inside the wall, negative (empty) outside.
    # The surface is where the field is 0.
    gyroid = (np.sin((2 * np.pi / period_x) * x) * np.cos((2 * np.pi / period_y) * y)
              + np.sin((2 * np.pi / period_y) * y) * np.cos((2 * np.pi / period_z) * z)
              + np.sin((2 * np.pi / period_z) * z) * np.cos((2 * np.pi / period_x) * x)) - 0.5
    field = thickness - np.abs(gyroid)
    field = np.minimum(field, distance_inside_box(x,y,z))
    return field


# define x,y,z axis with the number of points equal to the resolution along each axis
x1 = np.linspace(0, box_size_x, resolution_x)
y1 = np.linspace(0, box_size_y, resolution_y)
z1 = np.linspace(0, box_size_z, resolution_z)

x, y, z = np.meshgrid(x1, y1, z1, indexing="ij")

# compute the resulting spacing
coarse_spacing = np.array([x1[1] - x1[0], y1[1] - y1[0], z1[1] - z1[0]])

# Marching cubes needs several corners per gyroid period to follow the surface
if (period_x < 5 * coarse_spacing[0] or period_y < 5 * coarse_spacing[1]
        or period_z < 5 * coarse_spacing[2]):
    raise ValueError(f"Coarse cells {coarse_spacing} are too large for the period: "
                     f"use at least 5 cells per period along each axis.")

# compute the field
v = compute_field(x, y, z, period_x, period_y, period_z, thickness)

print(f"Coarse grid: {len(x1)} x {len(y1)} x {len(z1)}, "
      f" with a spacing of {coarse_spacing}")

triply.viz.twod_view_of_matrix(v, x1, y1, z1)
triply.viz.twod_view_of_matrix(distance_inside_box(x,y,z), x1, y1, z1)

In [ ]:
# The 8 corners of a cell, and the 8 children of a cell, as integer offsets 0 or 1 along each axis.
# Order: corner (a, b, c) is entry 4a + 2b + c, so 8 values reshape to a (2, 2, 2) block indexed [x, y, z].
corner_offsets = np.array([[a, b, c] for a in (0, 1) for b in (0, 1) for c in (0, 1)])   # shape (8, 3)

# flatten the initial grid: every cell of the coarse grid, is saved as integer indices (i, j, k)
flattened_grid = np.array([[i, j, k] for i in range(resolution_x-1)
                            for j in range(resolution_y-1)
                            for k in range(resolution_z-1)])  #shape (number of points (N), 3)

cell_size = coarse_spacing.copy()

for level in range(number_of_subdivisions + 1):
    # 1) organize the current grid into cells of 8 grid points
    # for every point in the flattened grid, add the 8 corner offsets, 
    cells_indices = flattened_grid[:, np.newaxis, :] + corner_offsets[np.newaxis, :, :] # (number of cells, 8, 3) # number of cells = len(flattened_grid)
                                                                                          # np.newaxis inserts an axis of length 1. When two arrays are combined, NumPy stretches a length-1 axis
                                                                                          # to the length of the other array's axis (broadcasting). Here (N, 1, 3) + (1, 8, 3) gives (N, 8, 3)
    # then multiply by the cell size to get the corner positions in mm
    cells_positions = cells_indices * cell_size        #shape (number of cells, 8, 3) 
    # compute the field at the 8 corners of every cell 
    cells_values = compute_field(cells_positions[..., 0], cells_positions[..., 1], cells_positions[..., 2],
                                  period_x, period_y, period_z, thickness)               # (number of cells, 8)

    # Keep only the cells crossing the surface: some corners solid, but not all
    corner_is_solid = cells_values > 0
    crosses_surface = corner_is_solid.any(axis=1) & ~corner_is_solid.all(axis=1) 

    # Also keep the cells with a corner close to the surface: the surface may dip into them between the corners.
    # "Close" shrinks with the cell size: the field changes by at most field_slope_bound per mm.
    field_slope_bound = 2 * np.sqrt(3) * 2 * np.pi / min(period_x, period_y, period_z)
    margin = near_surface_factor * field_slope_bound * cell_size.max()
    near_surface = np.abs(cells_values).min(axis=1) < margin 

    # Keep only the cells crossing the surface
    keep = crosses_surface | near_surface
    cells_indices = cells_indices[keep]

    # Finest level reached: stop, these are the cells to mesh
    if level == number_of_subdivisions:
      cells_values = cells_values[keep]
      cells_positions = cells_positions[keep]
      #return cells_indices, cells_positions, cells_values, cell_size
      break

    # 2) Make it finer there.
    # Corner (0, 0, 0) is entry 0 of corner_offsets, so [:, 0, :] is the first corner (i, j, k) of every kept cell
    kept_first_corners_indices = cells_indices[:, 0, :]      # (number of kept cells, 3)

    # each point (at coordinates (i, j, k)) fathers 7 news points  at coordinates (2i + a, 2j + b, 2k + c), with a, b, c = (0, 1).
    flattened_grid = 2 * kept_first_corners_indices[:, np.newaxis, :] + corner_offsets[np.newaxis, :, :]   # (number of kept cells, 8, 3)
    flattened_grid = flattened_grid.reshape(-1, 3)                                                           # (number of kept cells * 8, 3)
    cell_size = cell_size / 2
    # this does create a few extra points, but they will be removed in the next iteration when we keep only the cells crossing the surface.

# Result, all at the finest resolution (N = number of cells crossing the surface).
      # cells_values    :(N,8) array of the 8 corner values of every cell crossing the surface
      # cells_indices   : (N,8,3) array of the 8 corner positions of every cell crossing the surface)
      # cells_positions : (N,8,3) array of the 8 corner positions of every cell crossing the surface
      # cell_size       : (3,) array of the size of the cells at the finest resolution


## 5 — mesh it


In [ ]:
all_vertices = []
all_faces = []
number_of_vertices_so_far = 0
number_of_cells_meshed = 0

# mesh_from_matrix logs on every call: silence it during the loop
triply.set_log_level("CRITICAL")
try:
    for corner_positions, corner_values in zip(cells_positions, cells_values):
        # corner_positions: (8, 3) positions of this cell's 8 corners, in the order of corner_offsets
        # corner_values:    (8,)   field at those corners

        # No sign change at the corners: no surface in this cell (marching cubes would raise an error)
        if corner_values.min() > 0 or corner_values.max() < 0:
            continue
        
        # first create the cell's axis from the corner positions
        cell_x = np.array([corner_positions[0, 0], corner_positions[7, 0]])
        cell_y = np.array([corner_positions[0, 1], corner_positions[7, 1]])
        cell_z = np.array([corner_positions[0, 2], corner_positions[7, 2]])

        # The 8 values as a (2, 2, 2) block indexed [x, y, z]
        cell_block = corner_values.reshape(2, 2, 2)

        # pad_width=0: no caps around a single cell, otherwise the pieces could not be sewn together.
        # The vertices come back in physical coordinates (origin read from cell_x1, cell_y1, cell_z1).
        try:
            vertices, faces = triply.mesh_tools.mesh_from_matrix(
                cell_block, iso_level=0.0, algo_step_size=1,
                x=cell_x, y=cell_y, z=cell_z, pad_width=0)
        except RuntimeError:
            continue    # some corners exactly at 0 but no real crossing: nothing to mesh

        # Faces count this cell's vertices from 0: shift them to the global vertex list
        all_vertices.append(vertices)
        all_faces.append(faces + number_of_vertices_so_far)
        number_of_vertices_so_far += len(vertices)
        number_of_cells_meshed += 1
finally:
    triply.set_log_level("INFO")

print(f"{number_of_cells_meshed} of {len(cells_values)} cells contain surface")

## 6 — Sew the pieces into one mesh

A vertex on a face shared by two cells was computed by both cells. Merging identical vertices joins the pieces.

In [ ]:
# process=False keeps the duplicate vertices at creation, so the merge below is explicit
refined_mesh = trimesh.Trimesh(vertices=np.vstack(all_vertices), faces=np.vstack(all_faces), process=False)
print(f"Before merging: {len(refined_mesh.vertices)} vertices")

refined_mesh.merge_vertices()
refined_mesh.update_faces(refined_mesh.nondegenerate_faces())     # drop zero-area triangles
print(f"After merging:  {len(refined_mesh.vertices)} vertices, {len(refined_mesh.faces)} faces")

triply.viz.view_mesh(refined_mesh.faces, refined_mesh.vertices)

## 8 — Look at the refinement (diagnostic, small grids only)

One horizontal plane of corners at a time. Dots are the corners used for meshing (filled = solid, white = empty) plus the coarse corners, solid lines are the coarse cells, dotted squares are the cells of the last level split into 4, and the orange line is the true surface.

Coarse planes hold coarse and fine corners; the other planes only hold the fine corners of meshed cells. Like section 7, this builds a full fine grid, so keep it for small grids.

In [ ]:
# Coarse corner i is fine corner i * fine_per_coarse
fine_per_coarse = 2 ** number_of_subdivisions

# What we know at each fine corner: NaN = not shown, 0 = empty, 1 = solid.
# Fine corner (m_x, m_y, m_z) is at position (m_x, m_y, m_z) * cell_size (cell_size = finest cell size after the loop).
corner_state = np.full(((resolution_x - 1) * fine_per_coarse + 1,
                        (resolution_y - 1) * fine_per_coarse + 1,
                        (resolution_z - 1) * fine_per_coarse + 1), np.nan)

# Coarse corners: every fine_per_coarse-th fine corner, from the coarse field v
corner_state[::fine_per_coarse, ::fine_per_coarse, ::fine_per_coarse] = v > 0

# Corners of the meshed cells: their fine indices and values come straight from the loop, nothing is recomputed.
# cells_indices is (number of cells, 8, 3), cells_values is (number of cells, 8).
corner_state[cells_indices[..., 0], cells_indices[..., 1], cells_indices[..., 2]] = cells_values > 0

# Positions of the fine corners
fine_x1 = np.arange(corner_state.shape[0]) * cell_size[0]
fine_y1 = np.arange(corner_state.shape[1]) * cell_size[1]
fine_z1 = np.arange(corner_state.shape[2]) * cell_size[2]
fine_corner_x, fine_corner_y = np.meshgrid(fine_x1, fine_y1, indexing="ij")     # one plane of corners

# Dense grid used only to draw the true surface (field = 0) as a reference line
contour_x1 = np.linspace(0, box_size_x, 200)
contour_y1 = np.linspace(0, box_size_y, 200)
contour_x, contour_y = np.meshgrid(contour_x1, contour_y1, indexing="ij")

# First corner (i, j, k) of every meshed cell, in fine indices
cells_first_corner = cells_indices[:, 0, :]

TRACES_PER_PLANE = 4
figure = go.Figure()

for plane_index, plane_z in enumerate(fine_z1):
    show_first = (plane_index == 0)

    # 1) Dotted squares: the meshed cells that touch this plane.
    #    A cell whose first corner is at fine index k spans fine planes k and k + 1.
    touches_plane = (cells_first_corner[:, 2] <= plane_index) & (plane_index <= cells_first_corner[:, 2] + 1)
    outline_x, outline_y = [], []
    for i, j in np.unique(cells_first_corner[touches_plane, :2], axis=0):
        # Cell (i, j) spans fine corners i .. i + 1 along x and j .. j + 1 along y
        left, right = (i + np.arange(2)) * cell_size[0]
        bottom, top = (j + np.arange(2)) * cell_size[1]
        outline_x += [left, right, right, left, left, None]
        outline_y += [bottom, bottom, top, top, bottom, None]
    figure.add_trace(go.Scatter(x=outline_x, y=outline_y, mode="lines",
                                name="meshed cell", showlegend=show_first, visible=show_first,
                                line=dict(color="#7b8794", width=1, dash="dot"), hoverinfo="skip"))

    # 2) True surface on this plane
    surface_values = compute_field(contour_x, contour_y, np.full_like(contour_x, plane_z),
                                   period_x, period_y, period_z, thickness)
    figure.add_trace(go.Contour(x=contour_x1, y=contour_y1, z=surface_values.T,     # Contour wants rows = y
                                contours=dict(start=0, end=0, size=1, coloring="none"),
                                line=dict(color="#d9480f", width=2), showscale=False,
                                name="true surface", showlegend=show_first, visible=show_first,
                                hoverinfo="skip"))

    # 3) and 4) Corners of this plane where the field was evaluated
    state_on_plane = corner_state[:, :, plane_index]
    is_solid_corner = state_on_plane == 1
    is_empty_corner = state_on_plane == 0          # NaN (not shown) is neither, so it is not drawn
    figure.add_trace(go.Scatter(x=fine_corner_x[is_solid_corner], y=fine_corner_y[is_solid_corner],
                                mode="markers", name="solid corner",
                                showlegend=show_first, visible=show_first,
                                marker=dict(size=7, color="#2e8b57")))
    figure.add_trace(go.Scatter(x=fine_corner_x[is_empty_corner], y=fine_corner_y[is_empty_corner],
                                mode="markers", name="empty corner",
                                showlegend=show_first, visible=show_first,
                                marker=dict(size=7, color="white", line=dict(color="#555555", width=1))))

# Coarse cells: the same on every plane, drawn once
for x_value in x1:
    figure.add_shape(type="line", x0=x_value, x1=x_value, y0=y1[0], y1=y1[-1],
                     line=dict(color="black", width=1), layer="below")
for y_value in y1:
    figure.add_shape(type="line", x0=x1[0], x1=x1[-1], y0=y_value, y1=y_value,
                     line=dict(color="black", width=1), layer="below")

# Slider: step p shows only the traces of plane p
slider_steps = []
for plane_index, plane_z in enumerate(fine_z1):
    visibility = [False] * len(figure.data)
    for trace_number in range(TRACES_PER_PLANE):
        visibility[TRACES_PER_PLANE * plane_index + trace_number] = True
    kind = "coarse and fine corners" if plane_index % fine_per_coarse == 0 else "fine corners only"
    slider_steps.append(dict(method="update", label=f"{plane_z:.1f}",
                             args=[{"visible": visibility},
                                   {"title": f"Plane z = {plane_z:.1f}  ({kind})"}]))

figure.update_layout(
    title=f"Plane z = {fine_z1[0]:.1f}  (coarse and fine corners)",
    sliders=[dict(active=0, steps=slider_steps, currentvalue=dict(prefix="z = "), pad=dict(t=40))],
    xaxis=dict(title="x", range=[x1[0] - 2, x1[-1] + 2], showgrid=False, zeroline=False),
    yaxis=dict(title="y", range=[y1[0] - 2, y1[-1] + 2], showgrid=False, zeroline=False, scaleanchor="x"),
    width=750, height=780, plot_bgcolor="white", legend=dict(orientation="h", y=1.06))
figure.show()